# 03 shuffle

**目的**: shuffle（パーティションをまたいでデータを配り直す処理）が、いつ起きて、どれだけ高くつくのかを測る（60分）。Spark のチューニングの大半は「shuffle を減らす・小さくする」ことなので、ここがいちばん大事な回になる。

終わったときに説明できるようになること:

- どの操作が shuffle を起こすか。計画のどこを見れば分かるか
- ステージとは何か。なぜ shuffle のところで区切られるのか
- `spark.sql.shuffle.partitions` の意味と、値の決め方
- shuffle の量を減らす書き方。shuffle の回数を数える方法
- AQE（Adaptive Query Execution）が実行中に何を変えるか

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from labkit import check_plan, count_nodes, get_spark, large, materialize, observe, timed

spark = get_spark("03_shuffle", {"spark.sql.adaptive.enabled": "false"})

orders = large(spark, "orders")
customers = large(spark, "customers")

---

## 実験1: どの操作が shuffle を起こすか

各パーティションの中だけで完結する操作（narrow）は shuffle しない。別のパーティションにある行と突き合わせる必要がある操作（wide）は shuffle する。計画の中では、shuffle は `Exchange` という演算子として現れる。

先に予想を書いてから、下のセルで確かめる。

| 操作 | 予想（shuffle する / しない） | 結果 |
|---|---|---|
| filter | | |
| withColumn | | |
| groupBy + count | | |
| distinct | | |
| orderBy | | |
| repartition(8) | | |
| coalesce(1) | | |
| union（自分自身と） | | |
| dropDuplicates(["customer_id"]) | | |

In [ ]:
# TODO 1-1: それぞれの操作を、引数の df に対して書く（... を埋める。filter は例）
ops = {
    "filter":          lambda df: df.filter(F.col("status") == "completed"),
    "withColumn":      lambda df: ...,   # 例: ordered_at から年を取り出した列を足す
    "groupBy + count": lambda df: ...,   # status ごとの件数
    "distinct":        lambda df: ...,   # customer_id の重複をなくす（select してから distinct）
    "orderBy":         lambda df: ...,   # ordered_at で並べる
    "repartition(8)":  lambda df: ...,
    "coalesce(1)":     lambda df: ...,
    "union":           lambda df: ...,   # df と df 自身を union
    "dropDuplicates":  lambda df: ...,   # customer_id が同じ行を1行にする
}

In [ ]:
for name, op in ops.items():
    n = count_nodes(op(orders), "Exchange")
    print(f"{name:16s} {'shuffle する' if n else 'shuffle しない'}（Exchange {n} 個）")

✏️ **観察: 予想と違ったものは？ shuffle する操作に共通していることは何？**

（ここに書く）

---

## 実験2: shuffle がステージの境界になる

Spark はジョブを**ステージ**に区切って実行する。区切りは shuffle のところに入る。shuffle の手前のステージが結果をディスクに書き出し（shuffle write）、次のステージがそれを読む（shuffle read）。

In [ ]:
per_customer = orders.groupBy("customer_id").agg(F.count("*").alias("n_orders"))
per_customer.explain()

with observe(spark, "顧客ごとの注文数"):
    materialize(per_customer)

✏️ **観察: ステージはいくつ？ それぞれのタスク数は、何によって決まっている？ shuffle で書き出されたサイズは？**

（ここに書く）

計画の読み方（下から上へ）:

1. `FileScan` → `HashAggregate(... partial_count ...)`: 各パーティションの中で、先に顧客ごとに数えてしまう
2. `Exchange hashpartitioning(customer_id, 200)`: customer_id のハッシュで 200 個のパーティションに配り直す。ここがステージの境界
3. `HashAggregate(... count ...)`: 同じ顧客の途中結果が1つのパーティションに集まったので、合計して仕上げる

1で先に集計している（partial aggregation）のがポイントで、shuffle を通るのは「元の行」ではなく「パーティションごとの途中結果」になる。Spark UI の Stages タブで、同じ数字と、ステージごとの DAG の図が見られる。

---

## 実験3: spark.sql.shuffle.partitions

`Exchange hashpartitioning(customer_id, 200)` の 200 は、`spark.sql.shuffle.partitions` の既定値。shuffle した後のパーティション数（＝ 次のステージのタスク数）を決める。

✏️ **予想: この値を 2 / 8 / 200 / 2000 に変えると、実行時間はどうなる？ いちばん速いのは？**

（ここに書く）

In [ ]:
# TODO 3-1: spark.sql.shuffle.partitions を変えながら、同じ集計の時間を測る
for n in [2, 8, 200, 2000]:
    spark.conf.set(..., n)
    with timed(f"shuffle.partitions = {n}"):
        materialize(orders.groupBy("customer_id").agg(F.count("*").alias("n_orders")))

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", 200)  # 既定値に戻す

✏️ **観察: 結果を表にする。多すぎると遅くなるのはなぜ？ 逆に、本番の数百GBのデータで 8 にしたら何が起きそう？**

（ここに書く）

タスクには、1つ起動するごとに固定のコストがかかる。このデータは shuffle 後で数十MBしかないので、2000 個に分けるとタスクの起動と管理ばかりに時間を使う。逆に、大きいデータを少ないパーティションに詰めると、1タスクが抱えるデータがメモリに収まらず、ディスクへの退避（spill）やメモリ不足が起きる。

目安としてよく言われるのは「shuffle 後の1パーティションが 100〜200MB 程度」。既定の 200 がちょうど良いデータ量は、実はかなり限られている。

---

## 実験4: shuffle を小さくする、減らす

### 4-a. shuffle を通る量を減らす

「顧客ごとの最後の注文日時」を2通りで求める。

In [ ]:
# TODO 4-1: 顧客ごとの最後の注文日時を、2通りで書く
#   latest_by_window : ウィンドウ関数で、顧客ごとに ordered_at の新しい順に連番を振り、1番目の行だけ残す
#                      使うもの: Window.partitionBy(...).orderBy(F.desc(...)) / F.row_number().over(...)
#   latest_by_groupby: groupBy して max(ordered_at) を取る（列名は last_ordered_at）
w = ...
latest_by_window = ...
latest_by_groupby = ...

In [ ]:
check_plan(latest_by_window, has=["Window", "Exchange hashpartitioning(customer_id"])
check_plan(latest_by_groupby, has=["partial_max", "Exchange hashpartitioning(customer_id"], lacks=["Window"])

with observe(spark, "ウィンドウ関数"):
    materialize(latest_by_window)
with observe(spark, "groupBy + max"):
    materialize(latest_by_groupby)

✏️ **観察: どちらも customer_id で1回 shuffle している。shuffle で書き出されたサイズと実行時間は、何倍違う？ その差はどこから来ている？**

（ここに書く）

ウィンドウ関数は、元の行をすべて（全列ごと）shuffle してからパーティション内で並べ替える。groupBy は、shuffle の前に途中結果まで集計するので、通る行数も列数も少ない。

「最新の1行をまるごと欲しい」ならウィンドウ関数が必要だが、「最新の日時だけ欲しい」なら groupBy で足りる。SQL で `ROW_NUMBER() ... = 1` と `GROUP BY` + `MAX` のどちらを書くかは、Spark では実行コストの差としてそのまま表れる。

### 4-b. 同じキーなら、shuffle は1回で済む

In [ ]:
# 結合方式を SortMergeJoin に固定して、素の shuffle を見る（broadcast は 04_joins で扱う）
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

✏️ **予想: 顧客ごとの注文数（per_customer）を customers と customer_id で join する。計画に Exchange はいくつ出る？（groupBy で1つ、join の両側で1つずつ、と数えると3つ）**

（ここに書く）

In [ ]:
# TODO 4-2: per_customer と customers を customer_id で join する
joined = ...

In [ ]:
print("Exchange の数:", count_nodes(joined, "Exchange"))
joined.explain()

✏️ **観察: Exchange はいくつだった？ join の左側（per_customer 側）に、join のための Exchange はある？**

（ここに書く）

groupBy の時点で、データはすでに customer_id のハッシュで分割されている。join が必要とする分け方と同じなので、Spark はもう一度 shuffle することをしない。計画を読むときは、**今、データがどのキーで分割されているか**を追うと、shuffle の回数を数えられる。

In [ ]:
# TODO 4-3: joined を、さらに segment ごとに集計する（n_orders の合計を total_orders という列名で）
by_segment = ...

In [ ]:
print("Exchange の数:", count_nodes(by_segment, "Exchange"))
with observe(spark, "segment ごとの注文数"):
    rows = by_segment.collect()
print(*rows, sep="\n")

✏️ **観察: Exchange は1つ増えた？ なぜ今度は shuffle が必要になった？**

（ここに書く）

---

## 実験5: AQE を ON にする

ここまでは、素の挙動を見るために AQE（Adaptive Query Execution）を切っていた。AQE は、shuffle が終わるたびに実際のデータ量を見て、残りの計画を書き換える機能だ。Spark 3.2 以降は既定で ON になっている。

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")

histogram = (
    orders.groupBy("customer_id").agg(F.count("*").alias("n_orders"))
    .groupBy("n_orders").agg(F.count("*").alias("n_customers"))
)
print("=== 実行前 ===")
histogram.explain()

with observe(spark, "注文数ごとの顧客数（AQE あり）"):
    rows = histogram.collect()

print("=== 実行後 ===")
histogram.explain()

✏️ **観察: 実行前の計画は isFinalPlan=false、実行後は true になっている。Final Plan に増えた演算子は？ 2つ目・3つ目のステージのタスク数は、AQE なしなら 200 のはずだが、いくつになった？**

（ここに書く）

`AQEShuffleRead coalesced` は、「shuffle してみたら1パーティションが小さかったので、まとめて読むことにした」という意味。実験3で手で探した「ちょうど良いパーティション数」を、Spark が実行中に決めてくれている。

AQE が実行中に変えるのは主に3つ:

1. shuffle 後の小さいパーティションをまとめる（今見たもの）
2. 結合方式を切り替える（`04_joins` の実験4）
3. 偏ったパーティションを分割する（`04_joins` の実験5）

実務では AQE は ON のままにする。ただし、AQE が直してくれるのは shuffle の**後**の話で、そもそも shuffle する量や回数は、書いたコードで決まる。

---

## まとめ

✏️ **このノートブックで分かったことを、自分の言葉で3行にまとめる**

（ここに書く）

### 説明できるか確かめる

1. shuffle はなぜ高くつく？（何を、どこに書いて、どこへ運んでいる？）
2. 計画の中で shuffle を見つけるには？ ステージ数は計画から予想できる？
3. `spark.sql.shuffle.partitions` を決めるとき、何を基準にする？
4. groupBy のほうがウィンドウ関数より shuffle が小さくなるのはなぜ？
5. 「groupBy → 同じキーで join」で shuffle が増えないのはなぜ？
6. AQE があれば `spark.sql.shuffle.partitions` は気にしなくて良い？

In [ ]:
spark.stop()